In [2]:
import os

import torch
from torchsummary import summary

from Main import MODEL_CONFIG
from Model import UNet, WrappedUNet
from aberration import ZERNIKE_NUMS

model_config = MODEL_CONFIG
with torch.no_grad():
    device = torch.device(model_config["device"])
    model = UNet(
        T=model_config["T"],
        ch=model_config["channel"],
        ch_mult=model_config["channel_mult"],
        attn=model_config["attn"],
        nums_resblocks=model_config["num_res_blocks"],
        dropout=model_config["dropout"],
    )
    ckpt = torch.load(
        os.path.join(
            model_config["save_weight_dir"], model_config["test_load_weight"]
        ),
        map_location=device,
    )
    model.load_state_dict(ckpt)
    print("model load weight done")

    # 创建包装后的模型
wrapped_model = WrappedUNet(model, model_config["T"]).to("cuda:0").double()

# 使用 summary() 只传入 x 的形状（不需要传入 t）
summary(wrapped_model, input_size=(1, ZERNIKE_NUMS))

model load weight done
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
         Embedding-1                    [-1, 2]          12,800
            Linear-2                    [-1, 8]              24
             Swish-3                    [-1, 8]               0
            Linear-4                    [-1, 8]              72
     TimeEmbedding-5                    [-1, 8]               0
            Conv1d-6                [-1, 2, 64]               8
         GroupNorm-7                [-1, 2, 64]               4
             Swish-8                [-1, 2, 64]               0
            Conv1d-9                [-1, 2, 64]              14
            Swish-10                    [-1, 8]               0
           Linear-11                    [-1, 2]              18
        GroupNorm-12                [-1, 2, 64]               4
            Swish-13                [-1, 2, 64]               0
          Dropou